# Finetuning

In [21]:
! pip install groq langchain_huggingface faiss-cpu langchain_groq langchain_community

Prompting

In [22]:
import os
import groq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq


In [23]:

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

## RAG

Load Document

In [25]:
!pip install langchian faiss-cpu openai  tiktoken

In [28]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k")

print(dataset)
print(dataset["train"][0])

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})
{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [30]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

documents = []
for entry in dataset['train']:
    page_content = f"Instruction: {entry['instruction']}\nContext: {entry['context']}\nResponse: {entry['response']}"
    metadata = {
        "category": entry['category'],
        "instruction": entry['instruction']
    }
    documents.append(Document(page_content=page_content, metadata=metadata))

vector_db = FAISS.from_documents(
    documents,
    embeddings
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [31]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key="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",
    model_name="llama-3.3-70b-versatile"
)

def rag_query(query):
    docs = vector_db.similarity_search(query, k=3)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    prompt = f"""
Use only the provided context to answer the question.

Context:
{context}

Question:
{query}
"""

    response = llm.invoke(prompt)

    return response.content

In [35]:
print(rag_query("What causes rainbows to form?"))

Rainbows are formed by a combination of light, water droplets, and the angle of the sun. When light passes through the water droplets, it is refracted, or bent, and separated into its individual colors, creating the multicolored effect of a rainbow. The specific position of the sun, the water droplets, and the observer's line of sight all contribute to the formation of a rainbow.


# Fine Tuning(PEFT +LoRA)

In [36]:
!pip install transformers datasets peft accelerate bitsandbytes

In [37]:
#step 1 : Load model
from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig
import bitsandbytes as bnb
model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer=AutoTokenizer.from_pretrained(model_name)
#define the bitsandbytes config for 8 bit quantization
quantization_config=BitsAndBytesConfig(load_in_8bit=True)
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [38]:
#step 2 : Apply LoRA(PEFT)
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training
#preapre the model for k bit trining
model = prepare_model_for_kbit_training(model)
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model,lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [40]:
from datasets import load_dataset

dataset = load_dataset(
    "databricks/databricks-dolly-15k",
    split="train"
)

print(dataset[0])

{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [42]:
def format_example(example):

    if example["context"]:
        text = f"""
### Instruction:
{example['instruction']}

### Context:
{example['context']}

### Response:
{example['response']}
"""
    else:
        text = f"""
### Instruction:
{example['instruction']}

### Response:
{example['response']}
"""

    return {"text": text}

dataset = dataset.map(format_example)

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [43]:
#step 4: tokenization
def tokenize_function(example):
  return tokenizer(example["text"],truncation=True,padding="max_length")
tokenized_dataset=dataset.map(tokenize_function)

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [ ]:
#step 5: Training
from transformers import Trainer , TrainingArguments
#add labels to the tokenized_dataset for causal language mdelling
def add_labels_to_dataset(examples):
  examples['labels']=examples['input_ids']
  return examples
tokenized_dataset=tokenized_dataset.map(add_labels_to_dataset,batched=True)
training_args=TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=50
)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
trainer.train()

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
10,11.821269
20,8.886147
30,5.368760
40,2.339004
50,0.653915


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In [ ]:
#step 6: inferences
def generate_response(prompt):
  inputs=tokenizer(prompt,return_tensors="pt").to("cuda")
  outputs=model.generate(**inputs,max_new_tokens=100)
  return tokenizer.decode(outputs[0])

In [ ]:
print(generate_response("If a company earns ₹10 lakh and spends ₹7 lakh, what is its profit?"))